# PM2.5 Auditor Workflow

This notebook supports two modes:

- `USE_SAMPLE_DATA = True`: run the reproducible sample workflow
- `USE_SAMPLE_DATA = False`: use AQS and OpenAQ API data with your own keys


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/wweijia0107/EAE-483-Group-Project.git'
REPO_NAME = 'EAE-483-Group-Project'

def in_colab() -> bool:
    return 'google.colab' in sys.modules

def is_project_root(path: Path) -> bool:
    return (path / 'config.yaml').exists() and (path / 'src' / 'pm25_auditor').exists()

def find_base_dir(start: Path):
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if is_project_root(candidate):
            return candidate
    return None

BASE_DIR = find_base_dir(Path.cwd())

if BASE_DIR is None and in_colab():
    repo_path = Path('/content') / REPO_NAME
    if not repo_path.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_path)], check=True)
    BASE_DIR = repo_path

if BASE_DIR is None:
    raise FileNotFoundError(
        'Could not find the project root. In Colab, open this notebook from GitHub. '
        'Locally, open it from inside the repository.'
    )

SRC_DIR = BASE_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Running in Colab:', in_colab())
print('BASE_DIR =', BASE_DIR)
print('SRC_DIR added =', SRC_DIR)

In [ ]:
import calendar
import os
import time

import pandas as pd
import requests

from pm25_auditor.pipeline import (
    build_coverage_summary,
    build_data_quality_summary,
    build_training_pairs,
    clean_aqs_daily,
    clean_openaq_daily,
    combine_pm25_sources,
    ensure_directories,
    load_config,
    load_source_frames,
    plot_monthly_mean,
    plot_station_map,
    run_model_comparison,
    station_quality_metrics,
)

config = load_config(BASE_DIR / 'config.yaml')
config

## Mode Settings

Leave `USE_SAMPLE_DATA = True` for the default reproducible class-project version. Switch it to `False` only when you have your own API keys ready.

## API Setup Notes

If you want to run real API data instead of the sample data:

1. Keep this notebook inside the repository structure.
2. Create a `.env` file in the repository root.
3. Add your keys in this format:

```env
AQS_EMAIL=your_email_here
AQS_KEY=your_aqs_key_here
OPENAQ_KEY=your_openaq_key_here
```

4. Change `USE_SAMPLE_DATA = False` in the next code cell.
5. Run the notebook from the top.

For Colab, you can create the `.env` file with a code cell like this:

```python
from pathlib import Path

env_text = '''AQS_EMAIL=your_email_here
AQS_KEY=your_aqs_key_here
OPENAQ_KEY=your_openaq_key_here'''

(BASE_DIR / '.env').write_text(env_text, encoding='utf-8')
print('Created', BASE_DIR / '.env')
```

The default API settings below intentionally keep the request size small so the notebook is practical in Colab.

In [ ]:
USE_SAMPLE_DATA = True

YEAR = config['year']
STATE_CODE = config['state_code']
AQS_PM25_PARAM = config['aqs_pm25_param']
OPENAQ_PM25_ID = config['openaq_pm25_id']
BBOX = config['bbox']

# Keep API pulls small by default so Colab runs are practical.
AQS_MONTHS = [1]
OPENAQ_DATE_FROM = f'{YEAR}-01-01T00:00:00Z'
OPENAQ_DATE_TO = f'{YEAR}-01-31T23:59:59Z'
OPENAQ_MAX_LOCATIONS = 5

print('USE_SAMPLE_DATA =', USE_SAMPLE_DATA)

In [ ]:
def read_env_file(base_dir: Path) -> dict:
    env_path = base_dir / '.env'
    values = {}
    if not env_path.exists():
        return values
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        values[key.strip()] = value.strip()
    return values

def get_secret(name: str, env_map: dict) -> str:
    return os.getenv(name) or env_map.get(name, '')

def month_date_range(year: int, month: int):
    last_day = calendar.monthrange(year, month)[1]
    return f'{year}{month:02d}01', f'{year}{month:02d}{last_day:02d}'

def safe_get_json(url: str, params=None, headers=None, pause_seconds: float = 1.0, timeout: int = 180, retries: int = 3):
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=timeout)
            response.raise_for_status()
            payload = response.json()
            time.sleep(pause_seconds)
            return payload
        except Exception as exc:
            last_error = exc
            print(f'Attempt {attempt + 1} failed: {exc}')
            time.sleep(pause_seconds)
    raise last_error

API_COLUMNS = [
    'station_id',
    'pair_id',
    'timestamp_utc',
    'timestamp_local',
    'latitude',
    'longitude',
    'units',
    'value',
    'qa_flag',
    'expected_count',
    'observed_count',
    'percent_complete',
]

def empty_api_frame():
    return pd.DataFrame(columns=API_COLUMNS)

def fetch_aqs_api_frame(email: str, key: str, state_code: str, param_code: str, year: int, months):
    records = []
    for month in months:
        bdate, edate = month_date_range(year, month)
        print(f'Requesting AQS {bdate} to {edate}...')
        payload = safe_get_json(
            'https://aqs.epa.gov/data/api/dailyData/byState',
            params={
                'email': email,
                'key': key,
                'param': param_code,
                'bdate': bdate,
                'edate': edate,
                'state': state_code,
            },
            pause_seconds=1.0,
            timeout=180,
            retries=3,
        )
        print('AQS header:', payload.get('Header'))
        rows = payload.get('Data', [])
        print('AQS returned rows:', len(rows))
        for row in rows:
            state = str(row.get('state_code', '')).zfill(2)
            county = str(row.get('county_code', '')).zfill(3)
            site = str(row.get('site_number', '')).zfill(4)
            station_id = f'{state}-{county}-{site}'
            date_local = row.get('date_local')
            records.append(
                {
                    'station_id': station_id,
                    'pair_id': station_id,
                    'timestamp_utc': date_local,
                    'timestamp_local': date_local,
                    'latitude': row.get('latitude'),
                    'longitude': row.get('longitude'),
                    'units': row.get('units_of_measure', 'ug/m3'),
                    'value': row.get('arithmetic_mean'),
                    'qa_flag': 'none',
                    'expected_count': 1,
                    'observed_count': 1,
                    'percent_complete': 100,
                }
            )
    if not records:
        return empty_api_frame()
    return pd.DataFrame(records, columns=API_COLUMNS)

def fetch_openaq_api_frame(api_key: str, bbox: dict, date_from: str, date_to: str, parameter_id: int, max_locations: int):
    headers = {'X-API-Key': api_key}
    bbox_str = ','.join(str(v) for v in [bbox['min_lon'], bbox['min_lat'], bbox['max_lon'], bbox['max_lat']])
    locations_payload = safe_get_json(
        'https://api.openaq.org/v3/locations',
        params={
            'bbox': bbox_str,
            'parameters_id': parameter_id,
            'limit': max_locations,
            'page': 1,
        },
        headers=headers,
        pause_seconds=0.5,
    )
    records = []
    for location in locations_payload.get('results', []):
        location_id = location.get('id')
        coordinates = location.get('coordinates', {}) or {}
        sensors_payload = safe_get_json(
            f'https://api.openaq.org/v3/locations/{location_id}/sensors',
            headers=headers,
            pause_seconds=0.5,
        )
        for sensor in sensors_payload.get('results', []):
            parameter = sensor.get('parameter', {}) or {}
            if parameter.get('id') != parameter_id:
                continue
            sensor_id = sensor.get('id')
            days_payload = safe_get_json(
                f'https://api.openaq.org/v3/sensors/{sensor_id}/days',
                params={
                    'date_from': date_from,
                    'date_to': date_to,
                    'limit': 1000,
                    'page': 1,
                },
                headers=headers,
                pause_seconds=0.5,
            )
            for row in days_payload.get('results', []):
                period = row.get('period', {}) or {}
                start = period.get('datetimeFrom', {}) or {}
                coverage = row.get('coverage', {}) or {}
                parameter_row = row.get('parameter', {}) or {}
                flag_info = row.get('flagInfo', {}) or {}
                records.append(
                    {
                        'station_id': str(sensor_id),
                        'pair_id': str(sensor_id),
                        'timestamp_utc': start.get('utc'),
                        'timestamp_local': start.get('local'),
                        'latitude': coordinates.get('latitude'),
                        'longitude': coordinates.get('longitude'),
                        'units': parameter_row.get('units', parameter.get('units', 'ug/m3')),
                        'value': row.get('value'),
                        'qa_flag': 'flagged' if flag_info.get('hasFlags') else 'none',
                        'expected_count': coverage.get('expectedCount'),
                        'observed_count': coverage.get('observedCount'),
                        'percent_complete': coverage.get('percentComplete'),
                    }
                )
    if not records:
        return empty_api_frame()
    return pd.DataFrame(records, columns=API_COLUMNS)

def assign_pair_ids_by_nearest_aqs(aqs_df: pd.DataFrame, openaq_df: pd.DataFrame):
    if aqs_df.empty or openaq_df.empty:
        return aqs_df, openaq_df
    required_cols = {'station_id', 'latitude', 'longitude'}
    if not required_cols.issubset(aqs_df.columns) or not required_cols.issubset(openaq_df.columns):
        return aqs_df, openaq_df
    aqs_stations = aqs_df[['station_id', 'latitude', 'longitude']].drop_duplicates().dropna()
    openaq_stations = openaq_df[['station_id', 'latitude', 'longitude']].drop_duplicates().dropna()
    if aqs_stations.empty or openaq_stations.empty:
        return aqs_df, openaq_df
    pair_map = {}
    for _, openaq_station in openaq_stations.iterrows():
        distances = (
            (aqs_stations['latitude'] - openaq_station['latitude']) ** 2
            + (aqs_stations['longitude'] - openaq_station['longitude']) ** 2
        )
        nearest_station_id = aqs_stations.loc[distances.idxmin(), 'station_id']
        pair_map[openaq_station['station_id']] = nearest_station_id
    aqs_out = aqs_df.copy()
    openaq_out = openaq_df.copy()
    aqs_out['pair_id'] = aqs_out['station_id']
    openaq_out['pair_id'] = openaq_out['station_id'].map(pair_map).fillna(openaq_out['pair_id'])
    return aqs_out, openaq_out

def save_notebook_outputs(base_dir: Path, config: dict, aqs_clean: pd.DataFrame, openaq_clean: pd.DataFrame,
                          audit_table: pd.DataFrame, station_metrics: pd.DataFrame,
                          coverage_summary: pd.DataFrame, data_quality_summary: pd.DataFrame,
                          model_metrics: pd.DataFrame):
    dirs = ensure_directories(base_dir, config)
    processed_dir = dirs['processed_dir']
    reports_dir = dirs['reports_dir']
    figures_dir = dirs['figures_dir']
    aqs_clean.to_csv(processed_dir / 'aqs_clean.csv', index=False)
    openaq_clean.to_csv(processed_dir / 'openaq_clean.csv', index=False)
    audit_table.to_csv(processed_dir / 'audit_table.csv', index=False)
    station_metrics.to_csv(reports_dir / 'station_quality_metrics.csv', index=False)
    coverage_summary.to_csv(reports_dir / 'coverage_summary.csv', index=False)
    data_quality_summary.to_csv(reports_dir / 'data_quality_summary.csv', index=False)
    model_metrics.to_csv(reports_dir / 'model_metrics.csv', index=False)
    plot_monthly_mean(audit_table, figures_dir / 'monthly_mean_by_provider.png')
    plot_station_map(audit_table, figures_dir / 'station_map.png')
    return {
        'processed_dir': processed_dir,
        'reports_dir': reports_dir,
        'figures_dir': figures_dir,
    }


## Load Data

In sample mode, the notebook reads the included CSV files. In API mode, it looks for `AQS_EMAIL`, `AQS_KEY`, and `OPENAQ_KEY` in environment variables or in a local `.env` file created from `.env.example`.

In [ ]:
if USE_SAMPLE_DATA:
    aqs_raw, openaq_raw = load_source_frames(BASE_DIR, config)
else:
    env_map = read_env_file(BASE_DIR)
    aqs_email = get_secret('AQS_EMAIL', env_map)
    aqs_key = get_secret('AQS_KEY', env_map)
    openaq_key = get_secret('OPENAQ_KEY', env_map)
    if not all([aqs_email, aqs_key, openaq_key]):
        raise ValueError(
            'Missing API credentials. Add them to environment variables or create a .env file '
            'from .env.example before setting USE_SAMPLE_DATA = False.'
        )
    aqs_raw = fetch_aqs_api_frame(aqs_email, aqs_key, STATE_CODE, AQS_PM25_PARAM, YEAR, AQS_MONTHS)
    openaq_raw = fetch_openaq_api_frame(
        openaq_key,
        BBOX,
        OPENAQ_DATE_FROM,
        OPENAQ_DATE_TO,
        OPENAQ_PM25_ID,
        OPENAQ_MAX_LOCATIONS,
    )

print('AQS raw rows:', len(aqs_raw))
print('OpenAQ raw rows:', len(openaq_raw))
print('AQS columns:', list(aqs_raw.columns))
print('OpenAQ columns:', list(openaq_raw.columns))
display(aqs_raw.head())
display(openaq_raw.head())

## Clean, standardize, and pair sources

In [ ]:
if aqs_raw.empty:
    raise ValueError('AQS returned no rows. Try a different month, state, or API key setup.')
if openaq_raw.empty:
    raise ValueError('OpenAQ returned no rows. Try a wider bbox, more locations, or a different date range.')

aqs_clean = clean_aqs_daily(aqs_raw)
openaq_clean = clean_openaq_daily(openaq_raw)

if not USE_SAMPLE_DATA:
    aqs_clean, openaq_clean = assign_pair_ids_by_nearest_aqs(aqs_clean, openaq_clean)

audit_table = combine_pm25_sources(aqs_clean, openaq_clean)
station_metrics = station_quality_metrics(audit_table)
coverage_summary = build_coverage_summary(station_metrics)
data_quality_summary = build_data_quality_summary(audit_table, station_metrics)

display(audit_table.head())
display(station_metrics)
display(coverage_summary)
display(data_quality_summary)

## Calibration pairs and model metrics

In [ ]:
pairs = build_training_pairs(aqs_clean, openaq_clean)
display(pairs.head())
print('Training pairs:', len(pairs))

if pairs.empty or pairs['pair_id'].nunique() < 2:
    print('Not enough paired records for grouped model validation. Saving an empty model summary instead.')
    model_metrics = pd.DataFrame(
        [
            {
                'model_name': 'not_run',
                'folds': 0,
                'rmse_mean': None,
                'mae_mean': None,
                'r2_mean': None,
                'n_train_total': 0,
                'n_test_total': 0,
            }
        ]
    )
else:
    model_metrics = run_model_comparison(pairs)

display(model_metrics)

## Save outputs

In [ ]:
saved_paths = save_notebook_outputs(
    BASE_DIR,
    config,
    aqs_clean,
    openaq_clean,
    audit_table,
    station_metrics,
    coverage_summary,
    data_quality_summary,
    model_metrics,
)

saved_paths

In [ ]:
display(pd.read_csv(saved_paths['reports_dir'] / 'coverage_summary.csv'))
display(pd.read_csv(saved_paths['reports_dir'] / 'data_quality_summary.csv'))
display(pd.read_csv(saved_paths['reports_dir'] / 'model_metrics.csv'))

If you want to use real API data, keep this notebook in the repository root structure, create a `.env` file from `.env.example`, set `USE_SAMPLE_DATA = False`, and then rerun the notebook from the top.